# Comparing Syft output formats: SPDX, CycloneDX, GitHub, and native JSON

Syft turns a container image into a Software Bill of Materials (SBOM) and can write that SBOM out in several formats. The choice matters because every downstream consumer expects a specific schema: an image registry, a vulnerability scanner, or GitHub's dependency graph each want a different shape of document.

This notebook generates the same image's SBOM in four formats and compares their structure side by side:

- native JSON (Syft's own schema)
- SPDX JSON
- CycloneDX JSON
- GitHub JSON (the minimal format consumed by GitHub's dependency graph)

The goal is to see what is identical across formats (package name, version, PURL, licenses) and what is format-specific (SPDXID vs bom-ref, relationships vs dependencies).

In [ ]:
# last_verified: 2026-08-13 · syft (version n/a)
import json
import subprocess
from pathlib import Path

FORMATS = {
    'native':     'json',
    'spdx':       'spdx-json',
    'cyclonedx':  'cyclonedx-json',
    'github':     'github-json',
}

WORKDIR = Path('/tmp/syft-formats')
WORKDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def generate_all(image: str = 'node:18-alpine') -> dict:
    """Write the same image's SBOM in every format and return {name: parsed dict}."""
    out = {}
    for name, fmt in FORMATS.items():
        path = WORKDIR / f'{name}.json'
        cmd = ['syft', image, '-o', fmt, '--file', str(path)]
        print('running:', ' '.join(cmd))
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            if result.stdout:
                out[name] = json.loads(result.stdout)
            else:
                raise RuntimeError(f"syft ({fmt}) failed: {result.stderr}")
        else:
            out[name] = json.loads(path.read_text())
    return out

sboms = generate_all()
print('generated formats:', sorted(sboms))

In [ ]:
for name, doc in sboms.items():
    top = sorted(doc.keys())
    print(f'{name:10s} top-level keys: {top}')

In [ ]:
def package_collection(doc: dict):
    """Return the package list whichever schema the doc uses."""
    for key in ('artifacts', 'packages', 'components'):
        if key in doc:
            return key, doc[key]
    return None, []

for name, doc in sboms.items():
    key, items = package_collection(doc)
    print(f'{name:10s} collection key: {str(key):12s} count: {len(items)}')

In [ ]:
def sample_schema_fields(name: str, doc: dict) -> dict:
    """Show the per-format identity fields on the first package."""
    _, items = package_collection(doc)
    if not items:
        return {}
    pkg = items[0]
    def pick(*keys):
        for k in keys:
            if pkg.get(k) not in (None, [], '', {}):
                return pkg[k]
        return None
    return {
        'format': name,
        'name': pick('name'),
        'version': pick('version'),
        'purl': pick('purl', 'package_url', 'externalRefs'),
        'licenses': pick('licenses', 'licenseConcluded', 'licenseDeclared'),
    }

for name, doc in sboms.items():
    print(json.dumps(sample_schema_fields(name, doc), indent=2)[:500], '\n---')

## What the comparison shows

- **Native JSON** is Syft's own schema. It keeps the richest per-package metadata (file locations, layer indices, CPEs, metadata type) that the other formats compress away. Use it when the consumer is another tool in the Syft ecosystem or a custom pipeline that needs the full field set.
- **SPDX** is a full document (schema version, `documentNamespace`, `packages`, `relationships`) purpose-built for license and provenance exchange across tools. Every package gets an `SPDXID` that other documents can reference.
- **CycloneDX** organizes the same data as a BOM with `components` and `dependencies`, and identifies each component with a `bom-ref`. It is the format most scanners and registries expect for vulnerability correlation.
- **GitHub JSON** is deliberately minimal: it keeps just enough fields (name, version, PURL, licenses) to feed GitHub's dependency graph and drops the location/metadata detail.

The underlying packages are the same in all four; what changes is the envelope and how much per-package detail is retained.

## When to reach for each

A practical rule of thumb for choosing:

- Feed the **native** SBOM back into the Syft ecosystem, or debug cataloging locally.
- Publish **SPDX** when the SBOM is an artifact exchanged with other toolchains or auditors.
- Emit **CycloneDX** when the downstream job is vulnerability and risk correlation with scanners or registries.
- Write **GitHub JSON** when the SBOM uploads straight into GitHub's dependency graph.

The format choice is a contract between whoever generates the SBOM and whoever consumes it — every format describes the same image, only the schema and fidelity differ.